<a href="https://colab.research.google.com/github/BerkeleyExpertSystemTechnologiesLab/Squishy-Methane-Analysis/blob/jberry/Squish_Robot_Quant_Model_v6_5_Classes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Description

This model was produced by and for Squishy Robotics for the task of identifying and classifying methane leaks.


This model was made in conjunction with a synthetic dataset of 2 channel, 240 by 320 greyscale images of methane leaks
(2 x 240 x 320)
The first channel is a greyscale background image and the second channel is a greyscale gas plume image.



This model is experimental and uses the Optuna Hyperparameter Optimizer to search for successful hyperparameters (Learning Rate, Optimizer, Batch Size, Dropout %, etc...) and different optimizers. As such if you want to test a specific Model architecture you need to comment out the Optuna code and run a train/test on that specific model.

In [ ]:
# Optional: install dependencies in a fresh environment
# (Prefer managing dependencies via a requirements.txt / environment.yml)
# !pip -q install optuna


In [ ]:
from __future__ import annotations

import json
import logging
import os
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Iterable, List, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

import optuna

# -----------------------------
# Reproducibility + logging
# -----------------------------
LOGGER = logging.getLogger("squish_quant")
if not LOGGER.handlers:
    logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


def set_seed(seed: int = 42) -> None:
    """Best-effort deterministic behavior for experiments."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)


@dataclass(frozen=True)
class DataConfig:
    """Filesystem + label mapping config."""

    numpy_dir: Path = Path("./Final_Dataset/data")
    json_dir: Path = Path("./Final_Dataset/metadata")
    num_original_classes: int = 8
    num_classes: int = 5  # after mapping
    test_size: float = 0.2
    split_seed: int = 42


CFG = DataConfig()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOGGER.info(f"Using device: {DEVICE}")


In [ ]:
# Optional: extract dataset archive if present (more portable than shelling out to `unzip`)
zip_path = Path("Final_Dataset.zip")
out_dir = Path("Final_Dataset")

if zip_path.exists() and not out_dir.exists():
    import zipfile

    LOGGER.info(f"Extracting {zip_path} -> {out_dir}/")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)
elif not zip_path.exists():
    LOGGER.info("Final_Dataset.zip not found; skipping extraction.")
else:
    LOGGER.info("Final_Dataset already exists; skipping extraction.")


## Print out the structure of the data

In [ ]:
# Quick sanity check: show the shape/dtype of a sample frame (if available)
example_files = sorted(CFG.numpy_dir.glob("class_*/*.npy"))
if not example_files:
    LOGGER.warning(f"No .npy files found under {CFG.numpy_dir.resolve()}")
else:
    sample_path = example_files[0]
    sample = np.load(sample_path)
    LOGGER.info(f"Example file: {sample_path}")
    LOGGER.info(f"Shape: {sample.shape} | dtype: {sample.dtype}")
    LOGGER.info("Expected shape for GasVid frames is (2, 240, 320).")


In [ ]:
# Discover class folders (expects class_0 ... class_7 in the original dataset)
if not CFG.numpy_dir.exists():
    raise FileNotFoundError(f"Numpy directory not found: {CFG.numpy_dir.resolve()}")
classes = sorted([p.name for p in CFG.numpy_dir.iterdir() if p.is_dir()])
print(f"Classes found: {classes}")


## Create a dataset and dataloader

In [ ]:
class MultiModalDataset(Dataset):
    """Dataset returning (image_tensor, metadata_tensor, label_tensor).

    - image: FloatTensor of shape (C, H, W) (expected C=2)
    - metadata: FloatTensor of shape (num_metadata_features,)
    - label: LongTensor scalar
    """

    def __init__(
        self,
        numpy_files: Sequence[str | Path],
        json_files: Sequence[str | Path],
        labels: Sequence[int],
        *,
        metadata_keys: Sequence[str] = ("distance_m", "ppm"),
        cache_metadata: bool = False,
        warn_on_missing: bool = True,
        max_warnings: int = 10,
    ) -> None:
        if not (len(numpy_files) == len(json_files) == len(labels)):
            raise ValueError(
                "numpy_files, json_files, and labels must have the same length "
                f"({len(numpy_files)=}, {len(json_files)=}, {len(labels)=})"
            )

        self.numpy_files = [Path(p) for p in numpy_files]
        self.json_files = [Path(p) for p in json_files]
        self.labels = list(labels)

        self.metadata_keys = tuple(metadata_keys)
        self.cache_metadata = cache_metadata
        self.warn_on_missing = warn_on_missing
        self.max_warnings = max_warnings
        self._warning_count = 0

        self._cached_metadata: list[list[float]] | None = None
        if cache_metadata:
            self._cached_metadata = [self._load_metadata_features(p) for p in self.json_files]

    def __len__(self) -> int:
        return len(self.numpy_files)

    def __getitem__(self, idx: int):
        image_np = np.load(self.numpy_files[idx])
        image = torch.from_numpy(image_np).to(dtype=torch.float32)

        if self._cached_metadata is not None:
            meta_features = self._cached_metadata[idx]
        else:
            meta_features = self._load_metadata_features(self.json_files[idx])

        metadata = torch.tensor(meta_features, dtype=torch.float32)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return image, metadata, label

    def _load_metadata_features(self, json_path: Path) -> list[float]:
        with json_path.open("r", encoding="utf-8") as f:
            metadata = json.load(f)
        return self._extract_metadata_features(metadata)

    def _extract_metadata_features(self, metadata: dict) -> list[float]:
        """Extract numeric features from a metadata dict with safe defaults."""
        features: list[float] = []
        for key in self.metadata_keys:
            value = metadata.get(key, None)

            # Normalize missing / invalid values to 0.0 (and optionally warn a few times)
            if value is None or (isinstance(value, (int, float)) and float(value) == 0.0):
                if self.warn_on_missing and self._warning_count < self.max_warnings:
                    self._warning_count += 1
                    LOGGER.warning(
                        f"Metadata missing/invalid: {key}={value} in file; using 0.0"
                    )
                features.append(0.0)
            else:
                try:
                    features.append(float(value))
                except (TypeError, ValueError):
                    if self.warn_on_missing and self._warning_count < self.max_warnings:
                        self._warning_count += 1
                        LOGGER.warning(
                            f"Metadata non-numeric: {key}={value} in file; using 0.0"
                        )
                    features.append(0.0)

        return features


In [ ]:
# -----------------------------
# Label mapping (8 classes -> 5 classes)
# -----------------------------
CLASS_MAP_8_TO_5: dict[int, int] = {
    0: 0,
    1: 1,
    2: 1,
    3: 2,
    4: 2,
    5: 3,
    6: 3,
    7: 4,
}


def map_class(original_class: int) -> int:
    """Map the original GasVid 8-class labels to a 5-class scheme."""
    try:
        return CLASS_MAP_8_TO_5[original_class]
    except KeyError as e:
        raise ValueError(f"Unexpected original class: {original_class}") from e


In [ ]:
def build_file_index(
    cfg: DataConfig,
    label_map: Callable[[int], int] = map_class,
) -> tuple[list[Path], list[Path], list[int]]:
    """Collect aligned (npy_path, json_path, label) triplets.

    Assumptions (matching the original notebook):
    - Frames live under: cfg.numpy_dir / class_{k} / *.npy
    - Metadata lives under: cfg.json_dir / class_{k} / {video_id}_class_{k}.json
    - video_id is the prefix before the first underscore in the .npy filename.
    """
    all_numpy: list[Path] = []
    all_json: list[Path] = []
    all_labels: list[int] = []

    if not cfg.numpy_dir.exists():
        raise FileNotFoundError(f"Numpy directory not found: {cfg.numpy_dir.resolve()}")
    if not cfg.json_dir.exists():
        raise FileNotFoundError(f"Metadata directory not found: {cfg.json_dir.resolve()}")

    for class_idx in range(cfg.num_original_classes):
        numpy_class_dir = cfg.numpy_dir / f"class_{class_idx}"
        json_class_dir = cfg.json_dir / f"class_{class_idx}"

        numpy_files = sorted(numpy_class_dir.glob("*.npy"))
        LOGGER.info(f"class_{class_idx}: found {len(numpy_files)} frame files")

        for npy_path in numpy_files:
            base = npy_path.stem
            video_id = base.split("_")[0]

            json_path = json_class_dir / f"{video_id}_class_{class_idx}.json"
            if not json_path.exists():
                # Keep going; but log a warning (this used to print for every miss)
                LOGGER.warning(f"Missing JSON for {npy_path.name} -> {json_path.name}")
                continue

            all_numpy.append(npy_path)
            all_json.append(json_path)
            all_labels.append(label_map(class_idx))

    if not all_numpy:
        raise ValueError(
            "No samples found. Check that dataset paths are correct and extracted.\n"
            f"numpy_dir={cfg.numpy_dir.resolve()}\n"
            f"json_dir={cfg.json_dir.resolve()}"
        )

    LOGGER.info(f"TOTAL samples indexed: {len(all_numpy)}")
    dist = Counter(all_labels)
    LOGGER.info("Class distribution (after mapping): " + ", ".join([f"{k}:{v}" for k, v in sorted(dist.items())]))
    return all_numpy, all_json, all_labels


all_numpy_files, all_json_files, all_labels = build_file_index(CFG)


In [ ]:
def split_by_video(
    numpy_files: Sequence[Path],
    json_files: Sequence[Path],
    labels: Sequence[int],
    *,
    test_size: float,
    seed: int,
) -> tuple[list[Path], list[Path], list[int], list[Path], list[Path], list[int], list[str], list[str]]:
    """Split samples into train/test *by video_id* to avoid frame leakage."""

    video_to_indices: dict[str, list[int]] = defaultdict(list)
    for idx, npy_path in enumerate(numpy_files):
        video_id = npy_path.name.split("_")[0]
        video_to_indices[video_id].append(idx)

    video_ids = sorted(video_to_indices.keys())
    train_vids, test_vids = train_test_split(video_ids, test_size=test_size, random_state=seed)

    # Flatten indices
    train_indices = [i for vid in train_vids for i in video_to_indices[vid]]
    test_indices = [i for vid in test_vids for i in video_to_indices[vid]]

    # Create aligned splits
    train_numpy = [numpy_files[i] for i in train_indices]
    train_json = [json_files[i] for i in train_indices]
    train_labels = [labels[i] for i in train_indices]

    test_numpy = [numpy_files[i] for i in test_indices]
    test_json = [json_files[i] for i in test_indices]
    test_labels = [labels[i] for i in test_indices]

    # Sanity check: no overlap
    overlap = set(train_vids) & set(test_vids)
    if overlap:
        raise RuntimeError(f"Video leakage detected: {overlap}")
    return train_numpy, train_json, train_labels, test_numpy, test_json, test_labels, train_vids, test_vids


train_numpy, train_json, train_labels_list, test_numpy, test_json, test_labels_list, train_vids, test_vids = split_by_video(
    all_numpy_files,
    all_json_files,
    all_labels,
    test_size=CFG.test_size,
    seed=CFG.split_seed,
)
LOGGER.info(f"Train videos: {len(train_vids)} | Test videos: {len(test_vids)}")
LOGGER.info(f"Train samples: {len(train_numpy)} | Test samples: {len(test_numpy)}")


In [ ]:
def print_split_summary(train_labels: Sequence[int], test_labels: Sequence[int], num_classes: int) -> None:
    train_counts = Counter(train_labels)
    test_counts = Counter(test_labels)

    print("\n" + "=" * 70)
    print("DATASET SPLIT SUMMARY")
    print("=" * 70)

    def _fmt_counts(counts: Counter) -> str:
        total = sum(counts.values())
        lines = []
        for class_id in range(num_classes):
            c = counts.get(class_id, 0)
            pct = (c / total * 100) if total else 0.0
            lines.append(f"  Class {class_id}: {c:6d} ({pct:5.1f}%)")
        return "\n".join(lines)

    print(f"Train samples: {len(train_labels)}")
    print(_fmt_counts(train_counts))
    print("-" * 70)
    print(f"Test samples:  {len(test_labels)}")
    print(_fmt_counts(test_counts))
    print("=" * 70)

    missing_train = set(range(num_classes)) - set(train_counts.keys())
    missing_test = set(range(num_classes)) - set(test_counts.keys())
    if missing_train:
        LOGGER.warning(f"Training split missing classes: {sorted(missing_train)}")
    if missing_test:
        LOGGER.warning(f"Test split missing classes: {sorted(missing_test)}")


print_split_summary(train_labels_list, test_labels_list, num_classes=CFG.num_classes)


In [ ]:
train_dataset = MultiModalDataset(train_numpy, train_json, train_labels_list, cache_metadata=False)
test_dataset = MultiModalDataset(test_numpy, test_json, test_labels_list, cache_metadata=False)

# If metadata JSON I/O becomes a bottleneck, set cache_metadata=True above.


# Define the CNN model

## Define the Optuna Objective Function

This function will be called by Optuna for each trial. It will:
1. Suggest hyperparameters using the trial object.
2. Build and train the CNN model with the suggested hyperparameters.
3. Evaluate the model on a validation set
4. Return the metric to minimize (loss) or maximize (accuracy).

In [ ]:
# -----------------------------
# Model + training utilities
# -----------------------------
class ConvBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, dropout: float) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class VideoGasNet(nn.Module):
    """CNN over 2-channel frames + small MLP over metadata, fused for classification."""

    def __init__(
        self,
        *,
        num_metadata_features: int,
        num_classes: int,
        hidden_size: int = 128,
        fc_dropout: float = 0.3,
        cnn_dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.cnn = nn.Sequential(
            ConvBlock(2, 32, cnn_dropout),
            ConvBlock(32, 64, cnn_dropout),
            ConvBlock(64, 128, cnn_dropout),
        )
        # Avoid hard-coding flatten sizes (portable across resolutions)
        self.cnn_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.meta = nn.Sequential(
            nn.Linear(num_metadata_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(fc_dropout),
        )

        self.classifier = nn.Sequential(
            nn.Linear(128 + 64, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(inplace=True),
            nn.Dropout(fc_dropout),
            nn.Linear(hidden_size, num_classes),
        )

    def forward(self, image: torch.Tensor, metadata: torch.Tensor) -> torch.Tensor:
        x = self.cnn(image)
        x = self.cnn_pool(x).flatten(1)
        m = self.meta(metadata)
        z = torch.cat([x, m], dim=1)
        return self.classifier(z)


@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> tuple[float, float]:
    model.eval()
    total, correct = 0, 0
    total_loss = 0.0
    n_batches = 0

    for images, metadata, labels in loader:
        images = images.to(device, non_blocking=True)
        metadata = metadata.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images, metadata)
        loss = criterion(logits, labels)

        total_loss += float(loss.item())
        n_batches += 1

        preds = logits.argmax(dim=1)
        total += labels.size(0)
        correct += int((preds == labels).sum().item())

    avg_loss = total_loss / max(n_batches, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: optim.Optimizer,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float]:
    model.train()
    total, correct = 0, 0
    total_loss = 0.0
    n_batches = 0

    for images, metadata, labels in loader:
        images = images.to(device, non_blocking=True)
        metadata = metadata.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images, metadata)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item())
        n_batches += 1

        preds = logits.argmax(dim=1)
        total += labels.size(0)
        correct += int((preds == labels).sum().item())

    avg_loss = total_loss / max(n_batches, 1)
    acc = correct / max(total, 1)
    return avg_loss, acc


def make_optimizer(
    name: str,
    params: Iterable[torch.nn.Parameter],
    *,
    lr: float,
    weight_decay: float,
    momentum: float = 0.0,
) -> optim.Optimizer:
    name = name.lower()
    if name == "adam":
        return optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        return optim.SGD(params, lr=lr, momentum=momentum, weight_decay=weight_decay)
    raise ValueError(f"Unsupported optimizer: {name}")


NUM_METADATA_FEATURES = len(train_dataset.metadata_keys)
NUM_CLASSES = CFG.num_classes


def objective(trial: optuna.Trial) -> float:
    # Hyperparameters
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "AdamW", "SGD"])
    momentum = trial.suggest_float("momentum", 0.0, 0.95) if optimizer_name == "SGD" else 0.0
    weight_decay = trial.suggest_float("weight_decay", 0.0, 1e-2)
    hidden_size = trial.suggest_int("hidden_size", 64, 256, step=32)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    num_epochs = trial.suggest_int("num_epochs", 5, 10)
    fc_drop_rate = trial.suggest_float("fc_drop_rate", 0.1, 0.6)
    cnn_drop_rate = trial.suggest_float("cnn_drop_rate", 0.0, 0.3)

    # DataLoaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )
    val_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE.type == "cuda"),
    )

    # Model / optimizer
    model = VideoGasNet(
        num_metadata_features=NUM_METADATA_FEATURES,
        num_classes=NUM_CLASSES,
        hidden_size=hidden_size,
        fc_dropout=fc_drop_rate,
        cnn_dropout=cnn_drop_rate,
    ).to(DEVICE)

    optimizer = make_optimizer(
        optimizer_name,
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
        momentum=momentum,
    )
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, val_acc = evaluate(model, val_loader, criterion, DEVICE)

        # Let Optuna prune unpromising runs
        trial.report(val_acc, step=epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        best_val_acc = max(best_val_acc, val_acc)

        # Lightweight logging (avoids massive output per trial)
        if epoch == num_epochs - 1:
            LOGGER.info(
                f"Trial {trial.number} | val_acc={val_acc:.4f} | train_acc={train_acc:.4f} | "
                f"lr={lr:.2e} | opt={optimizer_name} | bs={batch_size} | hidden={hidden_size}"
            )

    return best_val_acc


## Run the Optuna Study

Now we will create an Optuna study and run the optimization process.

In [ ]:
# -----------------------------
# Optuna search
# -----------------------------
study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3),
)

study.optimize(objective, n_trials=30)

print("Best hyperparameters:", study.best_params)
print("Best validation accuracy:", study.best_value)

# Optional: visualization (requires plotly in many environments)
try:
    optuna.visualization.plot_param_importances(study).show()
except Exception as e:
    LOGGER.warning(f"Optuna visualization skipped: {e}")


#Sources:
###Hyperparameter Tuning with Optuna:
https://medium.com/@taeefnajib/hyperparameter-tuning-using-optuna-c46d7b29a3e

https://optuna.org/#code_examples
###Multi-Modal ML Models
https://www.nature.com/articles/s41598-025-14901-4
https://www.reddit.com/r/MachineLearning/comments/nziumg/combining_images_and_other_numeric_features_in_a/
https://pyimagesearch.com/2019/02/04/keras-multiple-inputs-and-mixed-data/

###Next Models to test:
VideoGasNet:
https://www.sciencedirect.com/science/article/pii/S0360544221017643

GasVit: https://www.sciencedirect.com/science/article/pii/S1568494623011560?via%3Dihub#sec3